# ICT Phase A — Transcript Extraction + Concept Citation Map

**Scope (source-material ingestion only):** fetch transcripts for a curated list of ICT YouTube
videos, normalize them to a structured timestamped schema, and produce a per-concept citation
map of raw timestamped excerpts. **No strategy logic, no edge claims, no interpretation or
summarization of what ICT "means."** Output is raw ICT wording with timestamps so a human can
read and cite it.

The reusable, network-free parsing/tagging logic lives in
`quant_research.utils.transcript_ingest` and is unit-tested in
`tests/utils/test_transcript_ingest.py`. This notebook is just orchestration.

## Dependencies (require operator approval before install)

- `youtube-transcript-api` — primary fetch (manual captions, then its generated track).
- `yt-dlp` (CLI) — fallback for auto-subs via `--write-auto-subs --skip-download`.

If neither is installed, the manifest/citation-map cells still run, but every video is logged
to `_missing.csv` because no transcript can be fetched. `whisper` is intentionally **not** used;
flag videos with no usable captions for a later decision.

## Pipeline

1. **Fetch** — `youtube-transcript-api` first, `yt-dlp` fallback. Idempotent (skip videos whose
   JSON already exists). Videos with no transcript go to `results/ict_citation_map/_missing.csv`
   and the run continues.
2. **Normalize** — one JSON per video at `data/processed/transcripts/<video_id>.json`.
3. **Citation map** — every concept lexicon scanned against every video (tags are metadata
   only), emitting `results/ict_citation_map/<concept>.json`. Pure keyword windowing with
   `±2` adjacent segments of context; no paraphrasing.

`data/` and `results/` are gitignored — outputs stay local.

In [1]:
"""Setup: locate the repo root, wire imports, and define output paths."""

import shutil
import sys
from importlib import util as importlib_util
from pathlib import Path

# Locate repo root by walking up to the directory that holds pyproject.toml.
_here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_here, *_here.parents) if (p / "pyproject.toml").is_file()),
    _here,
)
_src = REPO_ROOT / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from quant_research.utils import transcript_ingest as ti  # noqa: E402

MANIFEST_PATH = REPO_ROOT / "data" / "raw" / "ict_sources" / "manifest.csv"
TRANSCRIPT_DIR = REPO_ROOT / "data" / "processed" / "transcripts"
CITATION_DIR = REPO_ROOT / "results" / "ict_citation_map"
MISSING_CSV = CITATION_DIR / "_missing.csv"

TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
CITATION_DIR.mkdir(parents=True, exist_ok=True)

HAS_YT_TRANSCRIPT_API = importlib_util.find_spec("youtube_transcript_api") is not None
HAS_YT_DLP = shutil.which("yt-dlp") is not None

print(f"Repo root:            {REPO_ROOT}")
print(f"Manifest:             {MANIFEST_PATH}  (exists={MANIFEST_PATH.is_file()})")
print(f"Transcript out dir:   {TRANSCRIPT_DIR}")
print(f"Citation map out dir: {CITATION_DIR}")
print(f"youtube-transcript-api installed: {HAS_YT_TRANSCRIPT_API}")
print(f"yt-dlp CLI available:             {HAS_YT_DLP}")
if not (HAS_YT_TRANSCRIPT_API or HAS_YT_DLP):
    print(
        "\nNOTE: No fetch backend available. Install requires operator approval:\n"
        "  uv add youtube-transcript-api\n"
        "  uv add yt-dlp\n"
        "Until then, the fetch step logs every video to _missing.csv."
    )

Repo root:            C:\Users\chris\Documents\Trading\quant-research-lab
Manifest:             C:\Users\chris\Documents\Trading\quant-research-lab\data\raw\ict_sources\manifest.csv  (exists=True)
Transcript out dir:   C:\Users\chris\Documents\Trading\quant-research-lab\data\processed\transcripts
Citation map out dir: C:\Users\chris\Documents\Trading\quant-research-lab\results\ict_citation_map
youtube-transcript-api installed: True
yt-dlp CLI available:             True


## Load the manifest

Reads `data/raw/ict_sources/manifest.csv` (columns: `video_id, title, url, era, concept_tags`).
If the file is missing this stops with an error — we do **not** guess video IDs.

In [2]:
manifest_rows = ti.read_manifest(MANIFEST_PATH)  # raises FileNotFoundError if absent
print(f"Loaded {len(manifest_rows)} manifest rows.\n")
for row in manifest_rows[:5]:
    print(f"  {row.video_id}  era={row.era:<5}  tags={','.join(row.concept_tags)}  {row.title}")
if len(manifest_rows) > 5:
    print(f"  ... and {len(manifest_rows) - 5} more")

Loaded 41 manifest rows.

  kt6V4ai60fI  era=2022   tags=spec_a_sweep,spec_b_pd,spec_c_gap,settlement,session_timing  ICT Mentorship 2022 Introduction
  tmeCWULSTHc  era=2022   tags=spec_a_sweep,spec_b_pd,spec_c_gap,settlement,session_timing  2022 ICT Mentorship Episode 2
  nQfHZ2DEJ8c  era=2022   tags=spec_a_sweep,spec_b_pd,spec_c_gap,settlement,session_timing  2022 ICT Mentorship Episode 3
  L-ReMHiavPM  era=2022   tags=spec_a_sweep,spec_b_pd,spec_c_gap,settlement,session_timing  2022 ICT Mentorship Episode 4
  N29ZJ-o31xs  era=2022   tags=spec_a_sweep,spec_b_pd,spec_c_gap,settlement,session_timing  2022 ICT Mentorship Episode 5
  ... and 36 more


## Steps 1 + 2 — Fetch and normalize

For each manifest row: skip if its JSON already exists (idempotent); otherwise fetch
(`youtube-transcript-api` then `yt-dlp` fallback), normalize to the schema, and write
`data/processed/transcripts/<video_id>.json`. Videos with no retrievable transcript are
appended to `_missing.csv` and the run continues — one failure never aborts the batch.

In [3]:
import time

REQUEST_DELAY_S = 1.5  # polite pause between network fetches to avoid YouTube rate-limits

fetched_count = 0
skipped_count = 0
missing_count = 0

for row in manifest_rows:
    out_path = TRANSCRIPT_DIR / f"{row.video_id}.json"
    if out_path.exists():
        skipped_count += 1
        continue

    time.sleep(REQUEST_DELAY_S)
    try:
        result = ti.fetch_transcript(row.video_id)
    except Exception as exc:  # never abort the whole run on one failure
        ti.log_missing(MISSING_CSV, row, reason=f"fetch_error:{type(exc).__name__}")
        missing_count += 1
        print(f"  [fetch-error] {row.video_id}: {exc!r}")
        continue

    if result is None:
        ti.log_missing(MISSING_CSV, row, reason="no_transcript")
        missing_count += 1
        print(f"  [missing]     {row.video_id}: no retrievable transcript")
        continue

    source, raw_segments = result
    document = ti.normalize_transcript(
        video_id=row.video_id,
        title=row.title,
        url=row.url,
        era=row.era,
        concept_tags=row.concept_tags,
        source=source,
        raw_segments=raw_segments,
    )
    ti.write_json(out_path, document)
    fetched_count += 1
    print(f"  [{source:<9}] {row.video_id}: {len(document['segments'])} segments")

print(
    f"\nFetch summary: {fetched_count} written, {skipped_count} skipped (already present), "
    f"{missing_count} missing."
)
if missing_count:
    print(f"Missing videos logged to: {MISSING_CSV}")

  [missing]     j0Z2EbuJXGU: no retrievable transcript

Fetch summary: 0 written, 40 skipped (already present), 1 missing.
Missing videos logged to: C:\Users\chris\Documents\Trading\quant-research-lab\results\ict_citation_map\_missing.csv


## Step 3 — Concept citation map

Load every normalized transcript, scan **every** concept lexicon against **every** video
(`concept_tags` is metadata only and does not filter the scan), and write one
`results/ict_citation_map/<concept>.json` per concept. Each hit is a raw timestamped excerpt
with `±2` adjacent segments of context — pure keyword windowing, no summarization.

In [4]:
import json

transcripts = []
for json_path in sorted(TRANSCRIPT_DIR.glob("*.json")):
    with json_path.open("r", encoding="utf-8") as handle:
        transcripts.append(json.load(handle))

print(f"Loaded {len(transcripts)} normalized transcripts for scanning.\n")

citation_map = ti.build_citation_map(transcripts)  # default lexicon, ±2 context

for concept, hits in citation_map.items():
    ti.write_json(CITATION_DIR / f"{concept}.json", hits)
    n_videos = len({hit["video_id"] for hit in hits})
    print(f"  {concept:<16} {len(hits):>5} hits across {n_videos} video(s)")

print(f"\nCitation map written to: {CITATION_DIR}")

Loaded 40 normalized transcripts for scanning.



  spec_a_sweep       941 hits across 40 video(s)
  spec_b_pd          274 hits across 34 video(s)
  spec_c_gap         620 hits across 39 video(s)
  settlement           8 hits across 6 video(s)
  session_timing     334 hits across 32 video(s)

Citation map written to: C:\Users\chris\Documents\Trading\quant-research-lab\results\ict_citation_map
